# User Config

In [24]:
import pandas as pd
from collections import Counter
from pathlib import Path


LABEL_PATH = Path("./datasets/label.csv")
SPLIT_PATH = Path("./datasets/split.csv")

# CHANGE THIS to your ORIGINAL edge file path
EDGE_PATH = Path("./datasets/edge.csv")

# CHANGE THIS to your ORIGINAL user.json path
USER_JSON_PATH = Path("./datasets/user.json")

OUTPUT_DIR = Path("./datasets/new_sampling_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SAMPLED_USERS = OUTPUT_DIR / "sampled_users_20k.csv"
OUTPUT_SAMPLED_EDGES = OUTPUT_DIR / "sampled_internal_user_edges.csv"
OUTPUT_SAMPLED_USER_METADATA = OUTPUT_DIR / "sampled_user_metadata.csv"
OUTPUT_DIAGNOSTICS = OUTPUT_DIR / "sampling_diagnostics.csv"

RANDOM_STATE = 42
HUMAN_N = 12_200
BOT_N = 7_800

USER_USER_RELATIONS = {"following", "followers"}

df = pd.read_csv("./datasets/edge.csv")
df

,source_id,relation,target_id
0,u980749991491682304,followers,u1480979504696864775
1,u105387876,following,u402576793
2,u148520716,following,u59653593
3,u1276438425457967110,following,u1389155636693381120
4,u1445432327367237638,following,u848348952084828160
...,...,...,...
170185932,l40863046,membership,u1879831
170185933,l108800322,membership,u38142665
170185934,l994265798876303362,membership,u17759387
170185935,l82138286,membership,u932805374


# Load User Data

In [13]:
def load_label_split(label_path: Path, split_path: Path) -> pd.DataFrame:
    df_label = pd.read_csv(label_path)
    df_split = pd.read_csv(split_path)

    if not {"id", "label"}.issubset(df_label.columns):
        raise ValueError("label.csv must contain columns: id, label")

    if not {"id", "split"}.issubset(df_split.columns):
        raise ValueError("split.csv must contain columns: id, split")

    df_label["id"] = df_label["id"].astype(str)
    df_split["id"] = df_split["id"].astype(str)

    df_label["label"] = df_label["label"].astype(str).str.lower()
    df_split["split"] = df_split["split"].astype(str).str.lower()

    df_base = df_label.merge(df_split, on="id", how="left")
    return df_base

# Compute user degree from edge file

In [14]:
def compute_user_degree_from_edges(
    edge_path: Path,
    relations: set[str],
    chunksize: int = 1_000_000
) -> pd.DataFrame:
    degree_counter = Counter()

    for chunk in pd.read_csv(
        edge_path,
        chunksize=chunksize,
        usecols=["source_id", "target_id", "relation"]
    ):
        chunk["source_id"] = chunk["source_id"].astype(str)
        chunk["target_id"] = chunk["target_id"].astype(str)
        chunk["relation"] = chunk["relation"].astype(str)

        chunk = chunk[chunk["relation"].isin(relations)].copy()

        # keep only user-user edges
        chunk = chunk[
            chunk["source_id"].str.startswith("u") &
            chunk["target_id"].str.startswith("u")
        ]

        degree_counter.update(chunk["source_id"])
        degree_counter.update(chunk["target_id"])

    df_degree = pd.DataFrame({
        "id": list(degree_counter.keys()),
        "user_degree": list(degree_counter.values())
    })

    return df_degree

# Merge User Degree into User Table

In [15]:
def build_sampling_base(df_users: pd.DataFrame, df_degree: pd.DataFrame) -> pd.DataFrame:
    df = df_users.merge(df_degree, on="id", how="left")
    df["user_degree"] = df["user_degree"].fillna(0).astype(int)
    return df

# DEGREE-AWARE STRATIFIED SAMPLING

In [16]:
def degree_aware_stratified_sample(
    df: pd.DataFrame,
    human_n: int,
    bot_n: int,
    degree_col: str = "user_degree",
    random_state: int = 42
) -> pd.DataFrame:
    df = df.copy()

    # Handle case where too many zeros may make qcut unstable
    df["_degree_rank"] = df[degree_col].rank(method="first")

    df["degree_bucket"] = pd.qcut(
        df["_degree_rank"],
        q=5,
        labels=["very_low", "low", "mid", "high", "very_high"],
        duplicates="drop"
    )

    sampled_parts = []
    target_counts = {
        "human": human_n,
        "bot": bot_n
    }

    for label, target_n in target_counts.items():
        subset = df[df["label"] == label].copy()

        if len(subset) < target_n:
            raise ValueError(
                f"Not enough users for label '{label}'. "
                f"Requested {target_n}, available {len(subset)}."
            )

        bucket_dist = (
            subset["degree_bucket"]
            .value_counts(normalize=True, dropna=False)
            .sort_index()
        )

        bucket_targets = (bucket_dist * target_n).round().astype(int)

        diff = target_n - bucket_targets.sum()
        if diff != 0:
            largest_bucket = bucket_targets.idxmax()
            bucket_targets.loc[largest_bucket] += diff

        label_samples = []

        for bucket, n in bucket_targets.items():
            bucket_df = subset[subset["degree_bucket"] == bucket]
            n = min(n, len(bucket_df))

            if n > 0:
                label_samples.append(
                    bucket_df.sample(n=n, random_state=random_state)
                )

        sampled_label = pd.concat(label_samples, axis=0)

        # top up if rounding / bucket exhaustion caused shortfall
        if len(sampled_label) < target_n:
            already_selected = set(sampled_label["id"])
            remaining = subset[~subset["id"].isin(already_selected)]
            extra_n = target_n - len(sampled_label)

            extra = remaining.sample(n=extra_n, random_state=random_state)
            sampled_label = pd.concat([sampled_label, extra], axis=0)

        sampled_parts.append(sampled_label)

    df_sampled = (
        pd.concat(sampled_parts, axis=0)
        .drop(columns=["_degree_rank"])
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    return df_sampled

# FILTER INTERNAL USER-USER EDGES FOR SAMPLED USERS

# Diagnostics

In [18]:
def build_sampling_diagnostics(
    df_sampled_users: pd.DataFrame,
    df_sampled_edges: pd.DataFrame
) -> pd.DataFrame:
    sampled_ids = set(df_sampled_users["id"])

    connected_ids = set(df_sampled_edges["source_id"]) | set(df_sampled_edges["target_id"])
    isolated_ids = sampled_ids - connected_ids

    diag = pd.DataFrame([{
        "n_sampled_users": len(df_sampled_users),
        "n_humans": int((df_sampled_users["label"] == "human").sum()),
        "n_bots": int((df_sampled_users["label"] == "bot").sum()),
        "n_sampled_edges": len(df_sampled_edges),
        "n_connected_users": len(connected_ids),
        "n_isolated_users": len(isolated_ids),
        "isolate_ratio": len(isolated_ids) / max(len(sampled_ids), 1),
        "avg_user_degree_feature": float(df_sampled_users["user_degree"].mean()),
        "median_user_degree_feature": float(df_sampled_users["user_degree"].median()),
    }])

    return diag

In [ ]:
import pandas as pd
from pathlib import Path
from collections import Counter
import json


# =========================================================
# CONFIG
# =========================================================
LABEL_PATH = Path("./datasets/label.csv")
SPLIT_PATH = Path("./datasets/split.csv")

# CHANGE THIS to your ORIGINAL edge file path
EDGE_PATH = Path("./datasets/edge.csv")

# CHANGE THIS to your ORIGINAL user.json path
USER_JSON_PATH = Path("./datasets/user.json")

OUTPUT_DIR = Path("./datasets/new_sampling_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SAMPLED_USERS = OUTPUT_DIR / "sampled_users_20k.csv"
OUTPUT_INTERNAL_EDGES = OUTPUT_DIR / "sampled_internal_user_edges.csv"
OUTPUT_SAMPLED_USER_METADATA = OUTPUT_DIR / "sampled_user_metadata.csv"
OUTPUT_DIAGNOSTICS = OUTPUT_DIR / "sampling_diagnostics.csv"

RANDOM_STATE = 42
HUMAN_N = 12_200
BOT_N = 7_800

USER_USER_RELATIONS = {"following", "followers"}
USE_SPLIT_FILTER = None   # e.g. "train" if you only want train users

# Relations that indicate a user participates in the hetero graph
USER_RELEVANT_RELATIONS = {
    "following",
    "follower",
    "post",
    "pin",
    "like",
    "own",
    "member",
    "follow",
}

# Only for social-backbone diagnostics
USER_USER_RELATIONS = {
    "following",
    "follower",
}


# =========================================================
# LOAD LABEL + SPLIT
# =========================================================
def load_label_split(label_path: Path, split_path: Path) -> pd.DataFrame:
    df_label = pd.read_csv(label_path)
    df_split = pd.read_csv(split_path)

    if not {"id", "label"}.issubset(df_label.columns):
        raise ValueError("label.csv must contain columns: id, label")

    if not {"id", "split"}.issubset(df_split.columns):
        raise ValueError("split.csv must contain columns: id, split")

    df_label["id"] = df_label["id"].astype(str)
    df_split["id"] = df_split["id"].astype(str)

    # adjust this mapping if your labels are 0/1
    df_label["label"] = df_label["label"].astype(str).str.lower()
    df_split["split"] = df_split["split"].astype(str).str.lower()

    df_base = df_label.merge(df_split, on="id", how="left")
    return df_base


# =========================================================
# COMPUTE USER PARTICIPATION DEGREE
# =========================================================
def compute_user_participation_degree(
    edge_path: Path,
    relations: set[str],
    chunksize: int = 1_000_000
) -> pd.DataFrame:
    """
    Count how many user-relevant edges touch each user node.
    This is hetero-graph participation degree, not just user-user degree.
    """
    degree_counter = Counter()

    for chunk in pd.read_csv(
        edge_path,
        chunksize=chunksize,
        usecols=["source_id", "target_id", "relation"]
    ):
        chunk["source_id"] = chunk["source_id"].astype(str)
        chunk["target_id"] = chunk["target_id"].astype(str)
        chunk["relation"] = chunk["relation"].astype(str)

        chunk = chunk[chunk["relation"].isin(relations)].copy()

        # count any user appearing on either side
        source_users = chunk.loc[chunk["source_id"].str.startswith("u"), "source_id"]
        target_users = chunk.loc[chunk["target_id"].str.startswith("u"), "target_id"]

        degree_counter.update(source_users.tolist())
        degree_counter.update(target_users.tolist())

    df_degree = pd.DataFrame({
        "id": list(degree_counter.keys()),
        "participation_degree": list(degree_counter.values())
    })

    return df_degree


# =========================================================
# BUILD SAMPLING BASE
# =========================================================
def build_sampling_base(
    df_base: pd.DataFrame,
    df_degree: pd.DataFrame,
    split_filter: str | None = None
) -> pd.DataFrame:
    df = df_base.merge(df_degree, on="id", how="left")
    df["participation_degree"] = df["participation_degree"].fillna(0).astype(int)

    if split_filter is not None:
        df = df[df["split"] == split_filter].copy()

    # only users that actually appear in the retained hetero graph
    df = df[df["participation_degree"] > 0].copy()

    return df


# =========================================================
# DEGREE-AWARE STRATIFIED SAMPLING
# =========================================================
def degree_aware_stratified_sample(
    df: pd.DataFrame,
    human_n: int,
    bot_n: int,
    degree_col: str = "participation_degree",
    random_state: int = 42
) -> pd.DataFrame:
    df = df.copy()

    if len(df) == 0:
        raise ValueError("No users available for sampling.")

    df["_degree_rank"] = df[degree_col].rank(method="first")
    df["degree_bucket"] = pd.qcut(
        df["_degree_rank"],
        q=5,
        labels=["very_low", "low", "mid", "high", "very_high"],
        duplicates="drop"
    )

    sampled_parts = []
    target_counts = {
        "human": human_n,
        "bot": bot_n
    }

    for label, target_n in target_counts.items():
        subset = df[df["label"] == label].copy()

        if len(subset) < target_n:
            raise ValueError(
                f"Not enough {label} users after graph filtering. "
                f"Requested {target_n}, found {len(subset)}."
            )

        bucket_dist = (
            subset["degree_bucket"]
            .value_counts(normalize=True, dropna=False)
            .sort_index()
        )

        bucket_targets = (bucket_dist * target_n).round().astype(int)
        diff = target_n - bucket_targets.sum()
        if diff != 0:
            bucket_targets.loc[bucket_targets.idxmax()] += diff

        pieces = []
        for bucket, n in bucket_targets.items():
            bucket_df = subset[subset["degree_bucket"] == bucket]
            n = min(n, len(bucket_df))
            if n > 0:
                pieces.append(bucket_df.sample(n=n, random_state=random_state))

        sampled_label = pd.concat(pieces, axis=0) if pieces else pd.DataFrame(columns=subset.columns)

        # top up if short
        if len(sampled_label) < target_n:
            remaining = subset[~subset["id"].isin(sampled_label["id"])]
            extra_n = target_n - len(sampled_label)
            extra = remaining.sample(n=extra_n, random_state=random_state)
            sampled_label = pd.concat([sampled_label, extra], axis=0)

        sampled_parts.append(sampled_label)

    df_sampled = (
        pd.concat(sampled_parts, axis=0)
        .drop(columns=["_degree_rank"])
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    return df_sampled


# =========================================================
# FILTER INTERNAL EDGES FOR RETAINED NODES
# =========================================================
def filter_internal_edges_for_sample(
    edge_path: Path,
    sampled_user_ids: set[str],
    relations: set[str],
    chunksize: int = 1_000_000
) -> pd.DataFrame:
    """
    Keep all retained edges where at least one endpoint is a sampled user.
    This is for hetero-graph connectivity diagnostics and later graph construction.
    """
    kept = []

    for chunk in pd.read_csv(
        edge_path,
        chunksize=chunksize,
        usecols=["source_id", "target_id", "relation"]
    ):
        chunk["source_id"] = chunk["source_id"].astype(str)
        chunk["target_id"] = chunk["target_id"].astype(str)
        chunk["relation"] = chunk["relation"].astype(str)

        chunk = chunk[chunk["relation"].isin(relations)].copy()

        chunk = chunk[
            chunk["source_id"].isin(sampled_user_ids) |
            chunk["target_id"].isin(sampled_user_ids)
        ]

        if not chunk.empty:
            kept.append(chunk)

    if kept:
        return pd.concat(kept, ignore_index=True).drop_duplicates()

    return pd.DataFrame(columns=["source_id", "target_id", "relation"])


# =========================================================
# FILTER INTERNAL USER-USER EDGES
# =========================================================
def filter_internal_user_user_edges(
    edge_path: Path,
    sampled_user_ids: set[str],
    relations: set[str],
    chunksize: int = 1_000_000
) -> pd.DataFrame:
    kept = []

    for chunk in pd.read_csv(
        edge_path,
        chunksize=chunksize,
        usecols=["source_id", "target_id", "relation"]
    ):
        chunk["source_id"] = chunk["source_id"].astype(str)
        chunk["target_id"] = chunk["target_id"].astype(str)
        chunk["relation"] = chunk["relation"].astype(str)

        chunk = chunk[chunk["relation"].isin(relations)].copy()
        chunk = chunk[
            chunk["source_id"].isin(sampled_user_ids) &
            chunk["target_id"].isin(sampled_user_ids)
        ]

        if not chunk.empty:
            kept.append(chunk)

    if kept:
        return pd.concat(kept, ignore_index=True).drop_duplicates()

    return pd.DataFrame(columns=["source_id", "target_id", "relation"])


# =========================================================
# USER.JSON LOADING
# =========================================================
def load_user_json_as_df(json_path: Path) -> pd.DataFrame:
    if not json_path.exists():
        raise FileNotFoundError(f"user.json not found at: {json_path}")

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            df_user = pd.json_normalize(data)
        elif isinstance(data, dict):
            list_like_keys = [k for k, v in data.items() if isinstance(v, list)]
            if len(list_like_keys) == 1:
                df_user = pd.json_normalize(data[list_like_keys[0]])
            else:
                df_user = pd.json_normalize(data)
        else:
            raise ValueError("Unsupported JSON structure.")

    except json.JSONDecodeError:
        rows = []
        with open(json_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        df_user = pd.json_normalize(rows)

    possible_id_cols = ["id", "user_id", "author_id"]
    found = None
    for col in possible_id_cols:
        if col in df_user.columns:
            found = col
            break

    if found is None:
        raise ValueError(f"No user ID column found in user.json. Columns: {df_user.columns.tolist()}")

    if found != "id":
        df_user = df_user.rename(columns={found: "id"})

    df_user["id"] = df_user["id"].astype(str)
    df_user["id"] = df_user["id"].apply(lambda x: x if x.startswith("u") else f"u{x}")

    return df_user


def filter_user_metadata_to_sample(df_user: pd.DataFrame, sampled_user_ids: set[str]) -> pd.DataFrame:
    return df_user[df_user["id"].isin(sampled_user_ids)].copy()


# =========================================================
# DIAGNOSTICS
# =========================================================
def build_diagnostics(
    df_sampled: pd.DataFrame,
    df_hetero_edges: pd.DataFrame,
    df_user_user_edges: pd.DataFrame
) -> pd.DataFrame:
    sampled_ids = set(df_sampled["id"])

    # Hetero connectivity: sampled user appears in any retained edge
    hetero_connected_users = set()

    for row in df_hetero_edges.itertuples(index=False):
        if str(row.source_id).startswith("u") and row.source_id in sampled_ids:
            hetero_connected_users.add(row.source_id)
        if str(row.target_id).startswith("u") and row.target_id in sampled_ids:
            hetero_connected_users.add(row.target_id)

    hetero_isolated_users = sampled_ids - hetero_connected_users

    # Social connectivity: sampled user has an internal user-user edge
    social_connected_users = set(df_user_user_edges["source_id"]) | set(df_user_user_edges["target_id"])
    social_isolated_users = sampled_ids - social_connected_users

    return pd.DataFrame([{
        "n_sampled_users": len(df_sampled),
        "n_humans": int((df_sampled["label"] == "human").sum()),
        "n_bots": int((df_sampled["label"] == "bot").sum()),

        "n_hetero_edges_touching_sampled_users": len(df_hetero_edges),
        "n_hetero_connected_users": len(hetero_connected_users),
        "n_hetero_isolated_users": len(hetero_isolated_users),
        "hetero_isolate_ratio": len(hetero_isolated_users) / max(len(sampled_ids), 1),

        "n_internal_user_user_edges": len(df_user_user_edges),
        "n_social_connected_users": len(social_connected_users),
        "n_social_isolated_users": len(social_isolated_users),
        "social_isolate_ratio": len(social_isolated_users) / max(len(sampled_ids), 1),

        "avg_participation_degree": float(df_sampled["participation_degree"].mean()),
        "median_participation_degree": float(df_sampled["participation_degree"].median()),
    }])


# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading label + split...")
    df_base = load_label_split(LABEL_PATH, SPLIT_PATH)

    print("Computing user participation degree from hetero graph...")
    df_degree = compute_user_participation_degree(
        EDGE_PATH,
        USER_RELEVANT_RELATIONS
    )

    print("Building sampling base...")
    df_sampling = build_sampling_base(
        df_base,
        df_degree,
        split_filter=USE_SPLIT_FILTER
    )

    print("Running degree-aware stratified sampling...")
    df_sampled = degree_aware_stratified_sample(
        df_sampling,
        human_n=HUMAN_N,
        bot_n=BOT_N,
        degree_col="participation_degree",
        random_state=RANDOM_STATE
    )

    sampled_ids = set(df_sampled["id"])

    print("Filtering hetero edges touching sampled users...")
    df_hetero_edges = filter_internal_edges_for_sample(
        EDGE_PATH,
        sampled_ids,
        USER_RELEVANT_RELATIONS
    )

    print("Filtering internal user-user edges for diagnostics...")
    df_user_user_edges = filter_internal_user_user_edges(
        EDGE_PATH,
        sampled_ids,
        USER_USER_RELATIONS
    )

    print("Loading user.json and filtering to sampled users...")
    df_user = load_user_json_as_df(USER_JSON_PATH)
    df_sampled_user_metadata = filter_user_metadata_to_sample(df_user, sampled_ids)

    print("Building diagnostics...")
    df_diag = build_diagnostics(df_sampled, df_hetero_edges, df_user_user_edges)

    print("Saving outputs...")
    df_sampled.to_csv(OUTPUT_SAMPLED_USERS, index=False)
    df_hetero_edges.to_csv(OUTPUT_INTERNAL_EDGES, index=False)
    df_sampled_user_metadata.to_csv(OUTPUT_SAMPLED_USER_METADATA, index=False)
    df_diag.to_csv(OUTPUT_DIAGNOSTICS, index=False)

    print("\nDone.")
    print(f"Sampled users: {OUTPUT_SAMPLED_USERS}")
    print(f"Sampled retained edges: {OUTPUT_INTERNAL_EDGES}")
    print(f"Sampled user metadata: {OUTPUT_SAMPLED_USER_METADATA}")
    print(f"Diagnostics: {OUTPUT_DIAGNOSTICS}")
    print("\nDiagnostics preview:")
    print(df_diag.to_string(index=False))


if __name__ == "__main__":
    main()

Loading label + split...
Computing user participation degree from hetero graph...
